# Session 5 - From Cognitive Evaluation to Production Systems

**Core question:** When stimuli can change an LLM's behavior, how do we measure that effect reliably and use evidence responsibly in a real system?

This capstone connects the four project sessions to published research and a production design. The goal is not to claim that LLMs think like people; it is to make behavioral claims precise, testable, and operationally useful.

## Learning Outcomes

By the end of the lecture, students should be able to:

1. Explain how controlled stimuli and recent context can influence LLM outputs or judgments.
2. Distinguish a measured priming effect from a claim about human-like cognition.
3. Name key controls: lexical/semantic balance, randomization, model configuration, repetitions, and held-out evaluation.
4. Trace evidence through production monitoring, a release gate, and rollback.
5. Recommend whether a result is ready for further testing, inconclusive, or suitable for a guarded rollout.

## 1. Opening Question and Project Synthesis

**Warm-up :** If two users ask the same task but see different examples or sentence structures first, should we expect identical model behavior? What else besides the model could explain a difference?

| Session | What was built or studied | Evidence produced |
|---|---|---|
| 1: Stimuli | Controlled templates, balanced conditions, deterministic IDs and ordering | Validated stimulus set |
| 2: Agent loop | Prompt packaging, model calls, trial execution, artifact persistence | Trial-level inputs and outputs |
| 3: Analysis | Parsing, priming rates, effect sizes, statistical summaries | Quantified condition differences |
| 4: Optimization | Baselines, prompt variants, bandit exploration, critic, human gate | Recommendation with a trace |
| 5: Production | Versioned evaluation, monitoring, release controls, rollback | Responsible operational lifecycle |

 Ａn observed difference starts an explanation; it does not finish one. Preserve the exact stimulus, prompt version, model identifier, decoding settings, timestamp, and analysis version so another person can reconstruct the trial.

## 2. What Does Research Say About Stimuli and LLM Results?

A **stimulus** is controlled text or context presented to elicit a response. A prime can change recent structural context; examples can alter the in-context distribution; item order can affect judgments. These phenomena are related, but not interchangeable.

| Study | Design and reported result | Contribution |
|---|---|---|
| Sinclair, Jumelet, and Zuidema (2022), *Structural Persistence in Language Models* | Controlled sentence stimuli; reports structural priming in Transformer language models and uses it to probe abstract representations. | Direct precedent for a prime/target behavioral contrast. |
| Michaelov et al. (2023), *Structural Priming Demonstrates Abstract Grammatical Representations in Multilingual Language Models* | Compares models against eight crosslingual and four monolingual priming experiments; reports evidence for both forms of structural priming. | Effects can be tested across languages; stimulus construction remains central. |
| Lam et al. (2023), *Large Language Models Are Partially Primed in Pronoun Interpretation* | Replicates three psycholinguistic experiments in-context. InstructGPT adapts to local syntactic referential patterns, but not semantic biases; FLAN-UL2 did not show meaningful patterns. | Model and bias type matter; avoid generalizing from one model or stimulus family. |
| Cai et al. (2024), *Do Large Language Models Resemble Humans in Language Use?* | Tests ChatGPT and Vicuna on 12 preregistered experiments; human patterns reproduced in 10 and 7 experiments respectively, including reuse of recent structures, with divergences too. | Evidence for both convergence and non-convergence; one task cannot establish human equivalence. |
| Guan et al. (2025), *The Order Effect* (arXiv preprint) | Across paraphrasing, relevance judgment, and multiple-choice tasks, reports that shuffled input order can change accuracy; few-shot examples only partly mitigate effects. | Counterbalance and log order, especially in batch judgment systems. |
| Pucci, Li, and Sinclair (2026), *Patterns of Priming in Production* | Dative sentence-completion experiments report production priming, stronger effects with semantic coherence, and interaction among structural, lexical, and semantic alignment. | Balance meaning and lexical overlap, not syntax alone. |

### Careful Interpretation

These studies support the claim that input context and stimulus design can influence some measured LLM behaviors under specified conditions. They do **not** show that every model is primed in the same way, that a prompt effect reveals human-like mental states, or that results transfer automatically across tasks, models, languages, or deployments.

## Paper-by-Paper Results and Conclusions

The summary table above is a quick map. This section gives each paper's research question, method, result, conclusion, and a limitation to keep in mind. The papers study related but different phenomena, so do not treat them as one single measure of "priming."

### 1. Sinclair, Jumelet, and Zuidema (2022): Structural Persistence

[Structural Persistence in Language Models: Priming as a Window into Abstract Language Representations](https://direct.mit.edu/tacl/article-abstract/doi/10.1162/tacl_a_00504/113019), *Transactions of the Association for Computational Linguistics*.

- **Question and method:** Do Transformer language models reuse a syntactic structure after seeing that structure in a preceding sentence? The authors adapt structural-priming experiments to language models, using prime and target sentence stimuli to test persistence across sentence production.
- **Result:** The paper reports structural priming in the tested Transformer models: preceding structural context changes the relative likelihood of a later structural choice.
- **Conclusion:** Priming offers a behavioral probe of structural information carried across contexts. It makes the influence of sentence structure measurable rather than relying on intuitive prompt examples.
- **Limitation:** A priming effect is evidence of context-sensitive behavior, not by itself proof that a model has human-like grammatical representations or the same mechanism as a person. The result is bounded by the tested models, constructions, and prompt protocol.

### 2. Michaelov et al. (2023): Crosslingual Structural Priming

[Structural Priming Demonstrates Abstract Grammatical Representations in Multilingual Language Models](https://aclanthology.org/2023.emnlp-main.227/), EMNLP 2023.

- **Question and method:** Can structural priming reveal whether multilingual language models represent grammatical patterns beyond memorized words in one language? The study compares model behavior with eight crosslingual and four monolingual human priming experiments, covering six languages in crosslingual studies and three in monolingual studies.
- **Result:** The authors report evidence for monolingual and crosslingual structural priming. In tested conditions, a prime in one language can affect a structurally analogous choice in another.
- **Conclusion:** The results support the view that grammatical patterns in the evaluated multilingual models can generalize across languages and causally influence generated text.
- **Limitation:** Evidence comes from selected language pairs, structures, models, and elicitation procedures. Crosslingual priming does not establish that model representations are identical to human representations.

### 3. Lam et al. (2023): Pronoun Interpretation and Local Context

[Large Language Models Are Partially Primed in Pronoun Interpretation](https://aclanthology.org/2023.findings-acl.605/), Findings of ACL 2023.

- **Question and method:** Do LLMs adjust pronoun interpretation in response to recent referential patterns, as humans do? The authors adapt three psycholinguistic experiments into an in-context-learning setup.
- **Result:** InstructGPT adapted its pronominal interpretations to local syntactic patterns, but the reported adaptation did not extend to semantic biases. FLAN-UL2 did not produce meaningful patterns in those experiments.
- **Conclusion:** Some models show limited context adaptation, but it depends on the type of bias and on the model.
- **Limitation:** The word **partially** matters: this is not evidence that every model adapts or that syntactic and semantic context affect interpretation uniformly. Findings are bounded by the tested models and three paradigms.

### 4. Cai et al. (2024): Human-Language Behavior Across Tasks

[Do Large Language Models Resemble Humans in Language Use?](https://aclanthology.org/2024.cmcl-1.4/), CMCL 2024.

- **Question and method:** How often do conversational LLMs reproduce patterns from psycholinguistic experiments? The authors tested ChatGPT and Vicuna on 12 preregistered experiments spanning phenomena from sounds to dialogue.
- **Result:** ChatGPT reproduced the human pattern in 10 of 12 experiments; Vicuna did so in 7. Both showed some convergences, such as reusing recent sentence structures, but also failed to match humans on other phenomena, including preferences for shorter utterances and using context to resolve syntactic ambiguities.
- **Conclusion:** LLMs align with human experimental patterns on many language tasks while also showing systematic differences. Similarity must be assessed phenomenon by phenomenon.
- **Limitation:** Matching an experimental outcome does not prove shared cognitive processes. The counts apply to this set of 12 tasks and two model systems, not to human-likeness in general.

### 5. Guan et al. (2025): Input Order Effects

[The Order Effect: Investigating Prompt Sensitivity to Input Order in LLMs](https://arxiv.org/abs/2502.04134), arXiv preprint.

- **Question and method:** Does changing the order of information in prompts affect black-box LLM performance? The authors test input-order sensitivity in paraphrasing, relevance judgment, and multiple-choice tasks.
- **Result:** The preprint reports that input order significantly affects performance and that shuffled inputs can reduce accuracy. Few-shot prompting has mixed results and only partly mitigates the effect.
- **Conclusion:** Item order is a meaningful evaluation variable. Uncontrolled order can distort comparisons or batch decisions.
- **Limitation:** This is an arXiv preprint, not a peer-reviewed conference paper in the cited version. It tests selected models and tasks; it does not show that every ordering change harms every LLM task.

### 6. Chen et al. (2024): Threshold Priming in Relevance Judgments

[AI Can Be Cognitively Biased: An Exploratory Study on Threshold Priming in LLM-Based Batch Relevance Assessment](https://dl.acm.org/doi/10.1145/3673791.3698420), SIGIR-AP 2024.

- **Question and method:** Can relevance scores assigned earlier in a batch shift later LLM judgments? The study uses 10 topics from the TREC 2019 Deep Learning passage track and tests GPT-3.5, GPT-4, LLaMA 2 13B, and LLaMA 2 70B under varying score and batch-length conditions.
- **Result:** Across the tested combinations, later documents tended to receive lower scores after earlier documents got high relevance scores, and higher scores after earlier documents got low scores.
- **Conclusion:** Batch judgments can depend on preceding examples or decisions. Evaluation pipelines should examine ordering and batch composition, not only isolated prompts.
- **Limitation:** This is an exploratory IR study of specific topics, models, and scoring procedures. It does not establish the same bias size or direction in every domain or workflow.

### 7. Li et al. (2023): Directional Stimulus Prompting

[Guiding Large Language Models via Directional Stimulus Prompting](https://proceedings.neurips.cc/paper_files/paper/2023/hash/c5601d99ed028448f29d1dae2e4a926d-Abstract-Conference.html), NeurIPS 2023.

- **Question and method:** Can a smaller learned policy guide a black-box LLM without changing the LLM's weights? The method generates an instance-specific auxiliary hint, called a directional stimulus, and optimizes the policy using supervised data or rewards from model outputs.
- **Result:** The paper reports improvements on summarization, dialogue response generation, and chain-of-thought reasoning. In one MultiWOZ dialogue experiment, using 80 dialogues yielded a reported 41.4% relative improvement for ChatGPT; generated reasoning hints also improved InstructGPT accuracy over the compared general human-written and automatic prompts.
- **Conclusion:** Input stimuli can be deliberately optimized to steer outputs. Prompting can be measured and trained, rather than treated only as fixed wording chosen by hand.
- **Limitation:** This is a steering and optimization paper, not a syntactic-priming experiment. The relative gain is tied to a specific task and baseline; it is not a general 41.4% improvement across models or applications.

### 8. Pucci, Li, and Sinclair (2026): Priming in Production

[Patterns of Priming in Production: Lexical, Semantic and Structural Alignment in Language Model Generation](https://aclanthology.org/2026.findings-emnlp.1795/), Findings of EMNLP 2026.

- **Question and method:** Does structural priming appear when LMs generate sentence completions, and how does it relate to lexical and semantic alignment? The authors run controlled sentence-completion experiments on dative constructions.
- **Result:** They report production priming, especially when prime and target are semantically coherent. Double-object datives show a greater relative increase over baseline, while the more frequent prepositional-object construction shows the larger absolute increase. Structural priming is boosted by lexico-semantic coherence, and primed completions show more lexical repetition.
- **Conclusion:** Structural, lexical, and semantic alignment interact: priming can both facilitate and be facilitated by alignment at multiple linguistic levels.
- **Limitation:** Findings concern dative sentence completion and the tested models and stimuli. The observed interaction highlights a potential confound: an apparent syntax effect may partly travel with lexical or semantic repetition unless these are controlled.

### Synthesis Across the Papers

Together, these studies show that controlled input structure, local context, examples, and ordering can change measured LLM outputs or judgments. The effects are conditional on task, model, construction, semantics, and evaluation protocol. The strongest defensible conclusion is **stimuli matter and should be experimentally controlled**. The papers do not show that every model is affected identically, that an observed effect reveals a human-like mental state, or that results automatically transfer to another deployment.

### Guided Reading Questions 

1. What was manipulated: syntax, semantic coherence, discourse frequency, examples, or input order?
2. What was measured: a generated structure, a pronoun interpretation, task accuracy, or a score?
3. Which part is evidence, and which stronger claim would exceed that evidence?

**Facilitator note:** distinguish *stimulus sensitivity* (output changes with input), *mechanism* (why it changes), and *cognitive similarity* (whether the process is human-like). The first can be measured without establishing the last.

## 3. Turn a Stimulus Effect into a Trustworthy Test

| Design question | Practical control |
|---|---|
| What changed between conditions? | Change one intended factor at a time; keep task and target meaning stable. |
| Could words or meanings explain the effect? | Match or counterbalance lexical items and semantic plausibility; track overlap. |
| Could sequence explain it? | Randomize within runs and counterbalance order across blocks. |
| Could model variability explain it? | Pin model/API version and decoding parameters; repeat trials when stochasticity applies. |
| Are we tuning to the test set? | Separate development and held-out items; freeze the final set before comparison. |
| Is it practically important? | Report absolute difference, uncertainty, sample size, cost, latency, and failure rate. |
| Can the finding be reconstructed? | Store run/stimulus IDs, condition, prompt version, model/config, output, timestamp, parser and analysis versions. |

For binary outcomes, report condition rates and an effect estimate with uncertainty. Trials sharing an item, prompt, or model call may not be independent; use a design-appropriate analysis such as a mixed-effects logistic model when the data support it. Many generations from one exact prompt are not equivalent to many independent stimuli.

## 4. A Real Production System: Evidence Before Rollout

The diagram connects the live application path to the offline evaluation and release path. Offline testing compares a candidate against a versioned suite; only a reviewed candidate can enter a guarded canary. Production telemetry then checks whether the expected behavior persists under real traffic.

![Production LLM evaluation and release system](../artifacts/reports/session5-production-system.svg)

### Walkthrough

- **Live path:** use an approved, versioned prompt; validate model output; record only telemetry allowed by privacy policy.
- **Evaluation path:** run baseline and candidate over a fixed randomized suite; retain inputs, outputs, configuration, and failures as immutable trial records.
- **Decision path:** evaluate the target metric and guardrails (safety failures, malformed output, latency, cost). The critic may return *inconclusive*. A human approves release.
- **Rollout path:** begin with a small canary; compare against baseline; a guardrail breach rolls back to the last approved version.

**Production rule:** an offline win is evidence for a canary, not proof of safe behavior for every user or context. Keep an unmodified baseline, a rollback path, and an approval record.

### Map the Diagram Back to This Repository

| Production capability | Current project starting point | Extension needed for deployment |
|---|---|---|
| Stimulus set and validation | `src/stimuli/` | Versioned datasets and held-out suites |
| Experiment runner and persistence | `src/experiments/runner.py`, `src/agents/dialogue_runner.py` | Durable queue, retries, rate limits, idempotency |
| Analysis and recommendation | `src/analysis/`, `src/agents/analyst_agent.py` | Calibrated thresholds, uncertainty, guardrails |
| Prompt optimization | `src/optimization/` | Offline-only search; no self-promotion to production |
| Human approval | `src/agents/human_gate.py` | Authenticated approval, release ownership, audit trail |
| Artifacts and logs | `artifacts/runs/`, `src/common/logging_utils.py` | Retention, redaction, access policy, alerts |

This repository is a teaching prototype, not a production-ready service. Deployment also requires security, privacy, reliability, and operations engineering.




## Research Links and Suggested Reading

Read abstracts first; assign one paper per group and use the questions in Section 2. Sources were checked against publisher or conference pages where available.

1. Sinclair, A., Jumelet, J., and Zuidema, W. (2022). [Structural Persistence in Language Models: Priming as a Window into Abstract Language Representations](https://direct.mit.edu/tacl/article-abstract/doi/10.1162/tacl_a_00504/113019). *Transactions of the Association for Computational Linguistics*. DOI: [10.1162/tacl_a_00504](https://doi.org/10.1162/tacl_a_00504).
2. Michaelov, J. A., Arnett, C., Chang, T. A., and Bergen, B. K. (2023). [Structural Priming Demonstrates Abstract Grammatical Representations in Multilingual Language Models](https://aclanthology.org/2023.emnlp-main.227/). EMNLP 2023. DOI: [10.18653/v1/2023.emnlp-main.227](https://doi.org/10.18653/v1/2023.emnlp-main.227).
3. Lam, S.-Y., Zeng, Q., Zhang, K., You, C., and Voigt, R. (2023). [Large Language Models Are Partially Primed in Pronoun Interpretation](https://aclanthology.org/2023.findings-acl.605/). Findings of ACL 2023. DOI: [10.18653/v1/2023.findings-acl.605](https://doi.org/10.18653/v1/2023.findings-acl.605).
4. Cai, Z., Duan, X., Haslett, D., Wang, S., and Pickering, M. (2024). [Do Large Language Models Resemble Humans in Language Use?](https://aclanthology.org/2024.cmcl-1.4/). CMCL 2024. DOI: [10.18653/v1/2024.cmcl-1.4](https://doi.org/10.18653/v1/2024.cmcl-1.4).
5. Chen, N., Liu, J., Dong, X., Liu, Q., Sakai, T., and Wu, X.-M. (2024). [AI Can Be Cognitively Biased: An Exploratory Study on Threshold Priming in LLM-Based Batch Relevance Assessment](https://dl.acm.org/doi/10.1145/3673791.3698420). SIGIR-AP 2024. DOI: [10.1145/3673791.3698420](https://doi.org/10.1145/3673791.3698420).
6. Guan, B., Roosta, T., Passban, P., and Rezagholizadeh, M. (2025). [The Order Effect: Investigating Prompt Sensitivity to Input Order in LLMs](https://arxiv.org/abs/2502.04134). arXiv:2502.04134. Preprint; interpret accordingly.
7. Li, Z., Peng, B., He, P., Galley, M., Gao, J., and Yan, X. (2023). [Guiding Large Language Models via Directional Stimulus Prompting](https://proceedings.neurips.cc/paper_files/paper/2023/hash/c5601d99ed028448f29d1dae2e4a926d-Abstract-Conference.html). NeurIPS 2023; a complementary study of instance-specific prompts for steering behavior.
8. Pucci, G., Li, R., and Sinclair, A. (2026). [Patterns of Priming in Production: Lexical, Semantic and Structural Alignment in Language Model Generation](https://aclanthology.org/2026.findings-emnlp.1795/). Findings of EMNLP 2026. DOI: [10.18653/2026.findings-emnlp.1795](https://doi.org/10.18653/2026.findings-emnlp.1795).